# Andina Market · ETL en PySpark + Delta (Fabric o Databricks)

Mismo proceso que `python -m andina_etl`, escrito para Spark: **CSV → Bronze (Parquet crudo) → Silver (Delta, MERGE idempotente) → cuarentena → venta neta diaria (Delta)**.

**Cómo ejecutarlo**

- **Fabric:** crea un Lakehouse, sube los 4 CSV a `Files/andina/raw/`, adjunta el Lakehouse al notebook y ejecuta todas las celdas.
- **Databricks:** crea el catálogo/esquema `andina.datos` y un Volume con los CSV, cambia `PLATAFORMA` en la celda 1 y ejecuta todas las celdas.
- **Idempotencia:** vuelve a ejecutar todo. La celda de resumen debe mostrar 0 insertados y 0 actualizados.
- **Verificación:** la última celda compara contra las cifras del ETL local (19.270 pedidos y 20.124 pagos válidos; venta neta 14.177.333.600).

**Qué está probado y qué no**

- Probado con PySpark 3.5 local y ANSI activado: las reglas de calidad (celda 2) dan **los mismos conteos por regla, los mismos válidos y la misma venta neta** que la versión pandas/DuckDB sobre los datos reales.
- **No probado:** `MERGE INTO`, `DELETE` y `DESCRIBE HISTORY` de Delta, las rutas de Fabric/Databricks y la lectura `binaryFile`. Se escribieron con la sintaxis estándar de Delta, pero no se ejecutaron en ninguna plataforma. Si algo falla, probablemente sea ahí.
- **Caídas a mitad de corrida:** Delta no tiene transacciones entre tablas (la versión DuckDB sí usa una). Si el notebook falla después de escribir Silver y antes de Gold, vuelve a ejecutarlo con `RECALCULAR_GOLD = True` en la celda 1. Lo mismo tras cambiar la definición de venta.


In [ ]:
# === 1. Configuración: lo único que cambia entre Fabric y Databricks ===
import uuid
from datetime import datetime
from pyspark.sql import functions as F

PLATAFORMA = "fabric"            # "fabric" o "databricks"
FECHA_MIN, FECHA_MAX = "2025-01-01", "2026-09-30"   # rango válido de fecha_pedido
LOOKBACK_DIAS = 3                # ventana de reproceso de pagos sobre fecha_carga
ESTADO_VENTA = "ENTREGADO"       # lo único que cuenta como venta
RECALCULAR_GOLD = False          # True: recalcula todos los días (tras una caída o un cambio de definición)

if PLATAFORMA == "fabric":
    # Lakehouse por defecto adjunto al notebook
    RAW, BRONZE = "Files/andina/raw", "Files/andina/bronze"
    def T(nombre):               # tablas del Lakehouse: silver_pedidos, cuarentena, ...
        return nombre
else:
    # Unity Catalog + Volumes
    RAW, BRONZE = "/Volumes/andina/datos/raw", "/Volumes/andina/datos/bronze"
    def T(nombre):
        return f"andina.datos.{nombre}"


In [ ]:
# === 2. Reglas de calidad (espejo de la versión pandas; ver spark/rules_spark.py) ===
"""Reglas de calidad en PySpark. Espejo de andina_etl/rules.py (versión pandas/DuckDB).

Mismos códigos de regla, mismo criterio de duplicados, mismo hash de fila sobre los datos
crudos. Sirve para correr en Fabric o Databricks. No depende de Delta: son funciones puras
de DataFrame, por eso se pueden probar con PySpark local.

Todo el parseo usa try_cast / try_to_timestamp: con ANSI activado (por defecto en algunos
runtimes de Databricks) un valor sucio produciría una excepción en vez de NULL.
"""
from __future__ import annotations

from functools import reduce

from pyspark.sql import Column, DataFrame, Window
from pyspark.sql import functions as F

CANALES = {
    "WEB": "WEB", "PAGINA WEB": "WEB",
    "APP": "APP", "APLICACION": "APP",
    "TIENDA": "TIENDA", "TIENDA FISICA": "TIENDA",
}
ESTADOS_PEDIDO = ["ENTREGADO", "DEVUELTO", "CANCELADO"]
MEDIOS_PAGO = ["Tarjeta crédito", "PSE", "Billetera digital", "Contraentrega"]

COLS_CLIENTES = ["id_cliente", "nombre", "documento", "ciudad", "canal_registro", "fecha_registro"]
COLS_PEDIDOS = ["id_pedido", "id_cliente", "fecha_pedido", "canal", "categoria",
                "valor_bruto", "descuento", "estado", "fecha_entrega"]
COLS_PAGOS = ["id_pago", "id_pedido", "fecha_pago", "valor", "medio_pago", "fecha_carga"]


# ---------------------------------------------------------------- utilidades

def _txt(nombre: str) -> Column:
    """Texto sin espacios laterales; vacío -> NULL."""
    limpio = F.trim(F.col(nombre))
    return F.when(limpio == "", F.lit(None)).otherwise(limpio)


def _ts(nombre: str) -> Column:
    """Fecha o fecha-hora ISO; NULL si no se puede leer."""
    return F.coalesce(
        F.expr(f"try_to_timestamp({nombre}, 'yyyy-MM-dd HH:mm:ss')"),
        F.expr(f"try_to_timestamp({nombre}, 'yyyy-MM-dd')"),
    )


def canal_normalizado(nombre: str) -> Column:
    """'app ', 'APLICACION', 'Pagina Web' -> WEB / APP / TIENDA; NULL si no está en el mapa."""
    k = F.upper(F.trim(F.regexp_replace(
        F.translate(F.col(nombre), "ÁÉÍÓÚÜáéíóúü", "AEIOUUaeiouu"), r"\s+", " ")))
    expr = None
    for destino in ("WEB", "APP", "TIENDA"):
        cond = k.isin([c for c, d in CANALES.items() if d == destino])
        expr = F.when(cond, destino) if expr is None else expr.when(cond, destino)
    return expr


def hash_cols(columnas: list[str]) -> Column:
    """sha256 de las columnas crudas unidas con '|' (NULL = ''). Idéntico al de la versión pandas."""
    return F.sha2(F.concat_ws("|", *[F.coalesce(F.col(c), F.lit("")) for c in columnas]), 256)


def documento_limpio(nombre: str) -> Column:
    solo = F.regexp_replace(F.regexp_replace(F.col(nombre), r"\D", ""), r"^0+", "")
    return F.when(solo == "", F.lit(None)).otherwise(solo)


def _alguna(reglas: dict[str, Column]) -> Column:
    return reduce(lambda a, b: a | b, [F.coalesce(c, F.lit(False)) for c in reglas.values()])


def _cuarentena(df: DataFrame, tabla: str, cols_origen: list[str], reglas: dict[str, Column]) -> DataFrame:
    """Una fila por (registro, regla violada), con la fila original completa en JSON."""
    col_id = cols_origen[0]
    partes = [
        df.filter(F.coalesce(cond, F.lit(False))).select(
            F.lit(tabla).alias("tabla_origen"),
            F.coalesce(F.col(col_id), F.lit("")).alias("id_registro"),
            F.lit(codigo).alias("regla_violada"),
            F.col("row_hash"),
            F.to_json(F.struct(*cols_origen), {"ignoreNullFields": "false"}).alias("registro_json"),
        )
        for codigo, cond in reglas.items()
    ]
    return reduce(DataFrame.unionByName, partes)


def _con_veces_id(df: DataFrame, col_id: str) -> DataFrame:
    """Agrega x_veces_id: cuántas veces aparece el id en el lote. Va como columna porque Spark
    no admite funciones de ventana dentro de un filtro."""
    return df.withColumn("x_veces_id", F.count(F.lit(1)).over(Window.partitionBy(col_id)))


def _id_repetido(col_id: str) -> Column:
    """True si el id aparece más de una vez en el lote: todas las copias van a cuarentena."""
    return F.col(col_id).isNotNull() & (F.col("x_veces_id") > 1)


def _desempatar_duplicados(cand: DataFrame, col_id: str, existente: DataFrame) -> DataFrame:
    """Marca x_dup. Gana el que ya está en Silver; si ninguno está, el de menor id."""
    ganador_lote = F.min(col_id).over(Window.partitionBy("hash_negocio"))
    ex = existente.select("hash_negocio", F.col(existente.columns[1]).alias("x_ganador_silver"))
    return (cand.withColumn("x_ganador_lote", ganador_lote)
            .join(ex, "hash_negocio", "left")
            .withColumn("x_dup", F.col(col_id) != F.coalesce(F.col("x_ganador_silver"), F.col("x_ganador_lote"))))


# ------------------------------------------------------------------- clientes

def validar_clientes(raw: DataFrame) -> tuple[DataFrame, DataFrame]:
    df = (raw.withColumn("row_hash", hash_cols(COLS_CLIENTES))
          .withColumn("x_id", _txt("id_cliente"))
          .withColumn("x_fecha", F.expr("try_to_timestamp(fecha_registro, 'yyyy-MM-dd')")))
    n = F.row_number().over(Window.partitionBy("x_id").orderBy("row_hash"))
    df = df.withColumn("x_n", F.when(F.col("x_id").isNotNull(), n))
    reglas = {
        "ID_CLIENTE_NULO": F.col("x_id").isNull(),
        "FECHA_REGISTRO_INVALIDA": F.col("x_fecha").isNull(),
        "ID_CLIENTE_DUPLICADO": F.col("x_n") > 1,
    }
    cuarentena = _cuarentena(df, "clientes", COLS_CLIENTES, reglas)
    ok = df.filter(~_alguna(reglas))
    out = ok.select(
        F.col("x_id").alias("id_cliente"),
        _txt("nombre").alias("nombre"),
        F.col("documento").alias("documento_original"),
        documento_limpio("documento").alias("documento_limpio"),
        _txt("ciudad").alias("ciudad"),
        canal_normalizado("canal_registro").alias("canal_registro"),
        F.col("x_fecha").cast("date").alias("fecha_registro"),
        F.trunc(F.col("x_fecha"), "month").alias("cohorte_registro"),
        F.col("row_hash"),
    )
    return out, cuarentena


# -------------------------------------------------------------------- pedidos

def validar_pedidos(raw: DataFrame, fecha_min: str, fecha_max: str,
                    clientes_ids: DataFrame, negocio_existente: DataFrame) -> tuple[DataFrame, DataFrame]:
    """clientes_ids: DataFrame(id_cliente). negocio_existente: DataFrame(hash_negocio, id_pedido),
    una fila por hash (el id menor ya cargado en Silver)."""
    df = (raw
          .withColumn("row_hash", hash_cols(COLS_PEDIDOS))
          .withColumn("hash_negocio", hash_cols(["id_cliente", "fecha_pedido", "valor_bruto", "categoria"]))
          .withColumn("x_id", _txt("id_pedido"))
          .withColumn("x_cli", _txt("id_cliente"))
          .withColumn("x_canal", canal_normalizado("canal"))
          .withColumn("x_estado", F.upper(_txt("estado")))
          .withColumn("x_fp", _ts("fecha_pedido"))
          .withColumn("x_fe", _ts("fecha_entrega"))
          .withColumn("x_vb", F.expr("try_cast(valor_bruto as double)"))
          .withColumn("x_ds", F.expr("try_cast(descuento as double)"))
          .withColumn("x_dia", F.date_trunc("day", F.col("x_fp"))))
    df = _con_veces_id(df, "x_id")

    fmin, fmax = F.lit(fecha_min).cast("timestamp"), F.lit(fecha_max).cast("timestamp")
    c = F.col
    reglas = {
        "ID_PEDIDO_NULO": c("x_id").isNull(),
        "ID_PEDIDO_DUPLICADO": _id_repetido("x_id"),
        "CANAL_DESCONOCIDO": c("x_canal").isNull(),
        "ESTADO_INVALIDO": ~F.coalesce(c("x_estado").isin(ESTADOS_PEDIDO), F.lit(False)),
        "FECHA_PEDIDO_INVALIDA": c("x_fp").isNull(),
        "FECHA_FUERA_DE_RANGO": c("x_fp").isNotNull() & ((c("x_dia") < fmin) | (c("x_dia") > fmax)),
        "VALOR_BRUTO_NO_POSITIVO": c("x_vb").isNull() | (c("x_vb") <= 0),
        "DESCUENTO_INVALIDO": c("x_ds").isNull() | (c("x_ds") < 0) | (c("x_ds") > c("x_vb")),
        "ENTREGADO_SIN_FECHA": (c("x_estado") == "ENTREGADO") & c("x_fe").isNull(),
        "ENTREGA_ANTERIOR_A_PEDIDO": c("x_fe").isNotNull() & c("x_fp").isNotNull() & (c("x_fe") < c("x_dia")),
    }
    df = df.withColumn("x_malo", _alguna(reglas))
    cand = _desempatar_duplicados(df.filter(~c("x_malo")), "x_id", negocio_existente)

    cuarentena = _cuarentena(df, "pedidos", COLS_PEDIDOS, reglas).unionByName(
        _cuarentena(cand, "pedidos", COLS_PEDIDOS, {"DUPLICADO_LOGICO": c("x_dup")}))

    conocidos = clientes_ids.select(c("id_cliente").alias("x_conocido"), F.lit(True).alias("x_ok"))
    valido = (cand.filter(~c("x_dup"))
              .join(conocidos, c("x_cli") == c("x_conocido"), "left"))
    out = valido.select(
        c("x_id").alias("id_pedido"),
        c("x_cli").alias("id_cliente"),
        c("x_fp").alias("fecha_pedido"),
        c("x_dia").cast("date").alias("fecha_pedido_dia"),
        c("x_canal").alias("canal"),
        _txt("categoria").alias("categoria"),
        c("x_vb").cast("long").alias("valor_bruto"),
        c("x_ds").cast("long").alias("descuento"),
        (c("x_vb") - c("x_ds")).cast("long").alias("valor_neto"),
        c("x_estado").alias("estado"),
        c("x_fe").cast("date").alias("fecha_entrega"),
        c("x_ok").isNull().alias("flag_cliente_desconocido"),
        c("hash_negocio"),
        c("row_hash"),
    )
    return out, cuarentena


# ---------------------------------------------------------------------- pagos

def _fecha_pago() -> Column:
    """yyyy-MM-dd y dd/MM/yyyy con formato explícito: '03/04/2026' es 3 de abril."""
    return F.coalesce(
        F.when(F.col("fecha_pago").rlike(r"^\d{4}-\d{2}-\d{2}$"),
               F.expr("try_to_timestamp(fecha_pago, 'yyyy-MM-dd')")),
        F.when(F.col("fecha_pago").rlike(r"^\d{2}/\d{2}/\d{4}$"),
               F.expr("try_to_timestamp(fecha_pago, 'dd/MM/yyyy')")),
    )


def validar_pagos(raw: DataFrame, pedidos_dia: DataFrame, pedidos_en_cuarentena: DataFrame,
                  negocio_existente: DataFrame) -> tuple[DataFrame, DataFrame]:
    """pedidos_dia: DataFrame(id_pedido, fecha_pedido_dia) de Silver, un id por fila.
    pedidos_en_cuarentena: DataFrame(id_registro) con los ids de pedido rechazados."""
    c = F.col
    pd_ = pedidos_dia.select(c("id_pedido").alias("x_ped"), c("fecha_pedido_dia").cast("timestamp").alias("x_fped"))
    pq = pedidos_en_cuarentena.select(c("id_registro").alias("x_q"), F.lit(True).alias("x_en_q"))
    df = (raw
          .withColumn("row_hash", hash_cols(COLS_PAGOS))
          .withColumn("x_id", _txt("id_pago"))
          .withColumn("x_pedido", _txt("id_pedido"))
          .withColumn("x_fpago", _fecha_pago())
          .withColumn("x_fcarga", F.expr("try_to_timestamp(fecha_carga, 'yyyy-MM-dd')"))
          .withColumn("x_val", F.expr("try_cast(valor as double)"))
          .withColumn("x_medio", _txt("medio_pago"))
          .join(pd_, c("x_pedido") == c("x_ped"), "left")
          .join(pq, c("x_pedido") == c("x_q"), "left"))
    df = _con_veces_id(df, "x_id")
    en_silver = c("x_fped").isNotNull()
    en_q = F.coalesce(c("x_en_q"), F.lit(False))
    reglas = {
        "ID_PAGO_NULO": c("x_id").isNull(),
        "ID_PAGO_DUPLICADO": _id_repetido("x_id"),
        "ID_PEDIDO_NULO": c("x_pedido").isNull(),
        "FECHA_PAGO_INVALIDA": c("x_fpago").isNull(),
        "FECHA_CARGA_INVALIDA": c("x_fcarga").isNull(),
        "VALOR_INVALIDO": c("x_val").isNull() | (c("x_val") == 0),
        "MEDIO_PAGO_INVALIDO": ~F.coalesce(c("x_medio").isin(MEDIOS_PAGO), F.lit(False)),
        "PEDIDO_EN_CUARENTENA": c("x_pedido").isNotNull() & ~en_silver & en_q,
        "PAGO_HUERFANO": c("x_pedido").isNotNull() & ~en_silver & ~en_q,
        "PAGO_ANTERIOR_A_PEDIDO": en_silver & c("x_fpago").isNotNull() & (c("x_fpago") < c("x_fped")),
    }
    df = (df.withColumn("x_malo", _alguna(reglas))
          .withColumn("hash_negocio", F.sha2(F.concat_ws(
              "|", F.coalesce(c("x_pedido"), F.lit("")),
              F.coalesce(F.date_format(c("x_fpago"), "yyyy-MM-dd"), F.lit("")),
              F.coalesce(c("x_val").cast("long").cast("string"), F.lit("")),
              F.coalesce(c("x_medio"), F.lit(""))), 256)))
    cand = _desempatar_duplicados(df.filter(~c("x_malo")), "x_id", negocio_existente)

    cuarentena = _cuarentena(df, "pagos", COLS_PAGOS, reglas).unionByName(
        _cuarentena(cand, "pagos", COLS_PAGOS, {"DUPLICADO_LOGICO": c("x_dup")}))
    out = cand.filter(~c("x_dup")).select(
        c("x_id").alias("id_pago"),
        c("x_pedido").alias("id_pedido"),
        c("x_fpago").cast("date").alias("fecha_pago"),
        c("x_val").cast("long").alias("valor"),
        (c("x_val") < 0).alias("es_reembolso"),
        c("x_medio").alias("medio_pago"),
        c("x_fcarga").cast("date").alias("fecha_carga"),
        c("hash_negocio"),
        c("row_hash"),
    )
    return out, cuarentena


In [ ]:
# === 3. Tablas Delta (se crean una sola vez) ===
ddl = {
    "silver_clientes": """id_cliente STRING, nombre STRING, documento_original STRING, documento_limpio STRING,
        ciudad STRING, canal_registro STRING, fecha_registro DATE, cohorte_registro DATE,
        row_hash STRING, ingested_at TIMESTAMP""",
    "silver_pedidos": """id_pedido STRING, id_cliente STRING, fecha_pedido TIMESTAMP, fecha_pedido_dia DATE,
        canal STRING, categoria STRING, valor_bruto BIGINT, descuento BIGINT, valor_neto BIGINT,
        estado STRING, fecha_entrega DATE, flag_cliente_desconocido BOOLEAN, hash_negocio STRING,
        row_hash STRING, ingested_at TIMESTAMP""",
    "silver_pagos": """id_pago STRING, id_pedido STRING, fecha_pago DATE, valor BIGINT, es_reembolso BOOLEAN,
        medio_pago STRING, fecha_carga DATE, hash_negocio STRING, row_hash STRING, ingested_at TIMESTAMP""",
    "cuarentena": """tabla_origen STRING, id_registro STRING, regla_violada STRING, row_hash STRING,
        registro_json STRING, run_id STRING, fecha_proceso TIMESTAMP""",
    "etl_watermark": "tabla STRING, ultimo_valor DATE, ultimo_run_id STRING, actualizado_en TIMESTAMP",
    "bronze_manifest": "tabla STRING, sha256 STRING, ruta STRING, filas BIGINT, run_id STRING, ingested_at TIMESTAMP",
    "gold_venta_neta_diaria": """fecha DATE, canal STRING, categoria STRING, num_pedidos BIGINT,
        venta_bruta BIGINT, descuento BIGINT, venta_neta BIGINT""",
}
for tabla, columnas in ddl.items():
    spark.sql(f"CREATE TABLE IF NOT EXISTS {T(tabla)} ({columnas}) USING DELTA")


In [ ]:
# === 4. Bronze: copia cruda en Parquet, inmutable e idempotente ===
from pyspark.sql.types import LongType, StringType, StructField, StructType, TimestampType

SCHEMA_MANIFEST = StructType([
    StructField("tabla", StringType()), StructField("sha256", StringType()), StructField("ruta", StringType()),
    StructField("filas", LongType()), StructField("run_id", StringType()), StructField("ingested_at", TimestampType())])


def aterrizar(tabla, run_id):
    """Aterriza <tabla>.csv en Bronze si su contenido (sha256) es nuevo; devuelve lo que hay en Bronze.
    El mismo archivo dos veces no escribe nada; uno distinto crea un snapshot nuevo y no toca el anterior."""
    origen = f"{RAW}/{tabla}.csv"
    sha = spark.read.format("binaryFile").load(origen).select(F.sha2("content", 256)).first()[0]
    previo = (spark.table(T("bronze_manifest"))
              .filter((F.col("tabla") == tabla) & (F.col("sha256") == sha)).select("ruta").collect())
    if previo:
        ruta, nuevo = previo[0][0], False
    else:
        # todo como texto: tipar y validar es trabajo de Silver, no de Bronze
        df = spark.read.csv(origen, header=True, inferSchema=False)
        ahora = datetime.now()
        ruta = f"{BRONZE}/{tabla}/ingest_date={ahora:%Y-%m-%d}/{tabla}_{sha[:12]}"
        (df.withColumn("_ingested_at", F.lit(ahora)).withColumn("_run_id", F.lit(run_id))
           .withColumn("_source_file", F.lit(f"{tabla}.csv")).withColumn("_source_sha256", F.lit(sha))
           .write.mode("errorifexists").parquet(ruta))
        filas = spark.read.parquet(ruta).count()
        (spark.createDataFrame([(tabla, sha, ruta, filas, run_id, ahora)], SCHEMA_MANIFEST)
              .write.mode("append").saveAsTable(T("bronze_manifest")))
        nuevo = True
    leido = spark.read.parquet(ruta)
    leido = leido.select(*[c for c in leido.columns if not c.startswith("_")])
    return leido, nuevo


In [ ]:
# === 5. Clientes -> pedidos -> pagos (cada paso es idempotente por sí mismo) ===
run_id, inicio = str(uuid.uuid4()), datetime.now()
resumen = {}


def metricas_ultimo_merge(tabla):
    m = spark.sql(f"DESCRIBE HISTORY {T(tabla)} LIMIT 1").select("operationMetrics").first()[0]
    return int(m.get("numTargetRowsInserted", 0)), int(m.get("numTargetRowsUpdated", 0))


def merge_silver(df, tabla, clave):
    """Inserta lo nuevo y reemplaza lo que cambió de row_hash; lo idéntico no se toca."""
    df.withColumn("ingested_at", F.current_timestamp()).createOrReplaceTempView("stg")
    spark.sql(f"""
        MERGE INTO {T(tabla)} t USING stg s ON t.{clave} = s.{clave}
        WHEN MATCHED AND t.row_hash <> s.row_hash THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *""")
    return metricas_ultimo_merge(tabla)


def merge_cuarentena(df):
    """Una fila por (tabla, registro, regla, contenido). Reprocesar el mismo archivo no duplica filas."""
    (df.dropDuplicates(["tabla_origen", "id_registro", "regla_violada", "row_hash"])
       .withColumn("run_id", F.lit(run_id)).withColumn("fecha_proceso", F.current_timestamp())
       .createOrReplaceTempView("stg_q"))
    spark.sql(f"""
        MERGE INTO {T("cuarentena")} t USING stg_q s
        ON t.tabla_origen = s.tabla_origen AND t.id_registro = s.id_registro
           AND t.regla_violada = s.regla_violada AND t.row_hash = s.row_hash
        WHEN NOT MATCHED THEN INSERT *""")
    return int(spark.sql(f"DESCRIBE HISTORY {T('cuarentena')} LIMIT 1")
               .select("operationMetrics").first()[0].get("numTargetRowsInserted", 0))


# ---- clientes
cli_raw, nuevo_b = aterrizar("clientes", run_id)
cli_ok, cli_q = validar_clientes(cli_raw)
ins, upd = merge_silver(cli_ok, "silver_clientes", "id_cliente")
resumen["clientes"] = dict(insertados=ins, actualizados=upd, cuarentena_nueva=merge_cuarentena(cli_q), bronze_nuevo=nuevo_b)

# ---- pedidos: la fuente no trae fecha de carga, así que el incremental es por contenido (row_hash)
ped_raw, nuevo_b = aterrizar("pedidos", run_id)
negocio = (spark.table(T("silver_pedidos")).filter(F.col("hash_negocio").isNotNull())
           .groupBy("hash_negocio").agg(F.min("id_pedido").alias("id_pedido")))
ped_ok, ped_q = validar_pedidos(ped_raw, FECHA_MIN, FECHA_MAX, spark.table(T("silver_clientes")).select("id_cliente"), negocio)
ped_ok = ped_ok.cache()

# Días de venta a recalcular: el día nuevo de cada fila escrita y el día anterior si ya existía
previo = spark.table(T("silver_pedidos")).select("id_pedido", F.col("row_hash").alias("h_prev"), F.col("fecha_pedido_dia").alias("dia_prev"))
cambios = ped_ok.join(previo, "id_pedido", "left").filter(F.col("h_prev").isNull() | (F.col("h_prev") != F.col("row_hash")))
dias = sorted({r[0] for r in cambios.select("fecha_pedido_dia").union(cambios.select("dia_prev")).distinct().collect() if r[0]})


def rechazados(q, ok, clave):
    """Ids que fueron a cuarentena en esta corrida y no quedaron entre los válidos. Se recogen
    antes del MERGE: Spark es perezoso y, si no, se recalcularía contra el Silver ya modificado."""
    ids = q.select(F.trim("id_registro").alias(clave)).filter(F.col(clave) != "").distinct()
    return [r[0] for r in ids.join(ok.select(clave), clave, "left_anti").collect()]


def retirar(tabla, clave, ids):
    """Borra de Silver los registros que hoy son inválidos (ya quedaron en cuarentena)."""
    if not ids:
        return 0
    spark.createDataFrame([(i,) for i in ids], f"{clave} string").createOrReplaceTempView("ids_retiro")
    spark.sql(f"DELETE FROM {T(tabla)} WHERE {clave} IN (SELECT {clave} FROM ids_retiro)")
    return int(spark.sql(f"DESCRIBE HISTORY {T(tabla)} LIMIT 1")
               .select("operationMetrics").first()[0].get("numDeletedRows", 0))


ped_rechazados = rechazados(ped_q, ped_ok, "id_pedido")
ins, upd = merge_silver(ped_ok, "silver_pedidos", "id_pedido")
resumen["pedidos"] = dict(insertados=ins, actualizados=upd, cuarentena_nueva=merge_cuarentena(ped_q), bronze_nuevo=nuevo_b)
# Un pedido que ya estaba en Silver y hoy es inválido sale de Silver y deja de sumar venta.
dias_retirados = (spark.table(T("silver_pedidos")).filter(F.col("id_pedido").isin(ped_rechazados))
                  .select("fecha_pedido_dia").distinct().collect())
dias = sorted(set(dias) | {r[0] for r in dias_retirados})
resumen["pedidos"]["retirados"] = retirar("silver_pedidos", "id_pedido", ped_rechazados)

# ---- pagos: incremental por watermark de fecha_carga con ventana de reproceso
pag_raw, nuevo_b = aterrizar("pagos", run_id)
wm = spark.table(T("etl_watermark")).filter(F.col("tabla") == "pagos").select("ultimo_valor").first()
carga = F.expr("try_to_timestamp(fecha_carga, 'yyyy-MM-dd')")
if wm:
    # Los pagos que esperan a su pedido se releen siempre, aunque estén fuera de la ventana.
    pendientes = [r[0] for r in (
        spark.table(T("cuarentena"))
        .filter((F.col("tabla_origen") == "pagos") & F.col("regla_violada").isin("PAGO_HUERFANO", "PEDIDO_EN_CUARENTENA"))
        .select(F.trim("id_registro").alias("id_pago"))
        .join(spark.table(T("silver_pagos")), "id_pago", "left_anti")
        .union(spark.table(T("silver_pagos")).join(spark.table(T("silver_pedidos")), "id_pedido", "left_anti")
               .select("id_pago"))
        .distinct().collect())]
    pag_raw = pag_raw.filter(carga.isNull() | (carga >= F.expr(f"date_sub(DATE'{wm[0]}', {LOOKBACK_DIAS})"))
                             | F.trim("id_pago").isin(pendientes))
pag_raw = pag_raw.cache()
pedidos_dia = spark.table(T("silver_pedidos")).select("id_pedido", "fecha_pedido_dia")
ped_en_q = spark.table(T("cuarentena")).filter(F.col("tabla_origen") == "pedidos").select("id_registro").distinct()
negocio_pag = (spark.table(T("silver_pagos")).filter(F.col("hash_negocio").isNotNull())
               .groupBy("hash_negocio").agg(F.min("id_pago").alias("id_pago")))
pag_ok, pag_q = validar_pagos(pag_raw, pedidos_dia, ped_en_q, negocio_pag)
pag_ok = pag_ok.cache()
pag_rechazados = rechazados(pag_q, pag_ok, "id_pago")
ins, upd = merge_silver(pag_ok, "silver_pagos", "id_pago")
resumen["pagos"] = dict(insertados=ins, actualizados=upd, cuarentena_nueva=merge_cuarentena(pag_q), bronze_nuevo=nuevo_b)
resumen["pagos"]["retirados"] = retirar("silver_pagos", "id_pago", pag_rechazados)

maximo = pag_raw.select(F.max(carga).cast("date")).first()[0]
if maximo:
    nuevo_wm = max(maximo, wm[0]) if wm else maximo
    spark.sql(f"""
        MERGE INTO {T("etl_watermark")} t
        USING (SELECT 'pagos' AS tabla, DATE'{nuevo_wm}' AS ultimo_valor, '{run_id}' AS ultimo_run_id,
                      current_timestamp() AS actualizado_en) s
        ON t.tabla = s.tabla
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *""")


In [ ]:
# === 6. Venta neta diaria: se recalcula solo para los días tocados (depende solo de Silver) ===
if RECALCULAR_GOLD:
    dias = sorted({r[0] for r in spark.table(T("silver_pedidos")).select("fecha_pedido_dia")
                   .union(spark.table(T("gold_venta_neta_diaria")).select("fecha")).distinct().collect()})
if dias:
    lista = ", ".join(f"DATE'{d}'" for d in dias)
    spark.sql(f"DELETE FROM {T('gold_venta_neta_diaria')} WHERE fecha IN ({lista})")
    spark.sql(f"""
        INSERT INTO {T('gold_venta_neta_diaria')}
        SELECT fecha_pedido_dia, canal, categoria, count(*), sum(valor_bruto), sum(descuento), sum(valor_neto)
        FROM {T('silver_pedidos')}
        WHERE estado = '{ESTADO_VENTA}' AND fecha_pedido_dia IN ({lista})
        GROUP BY fecha_pedido_dia, canal, categoria""")
resumen["dias_recalculados"] = len(dias)

for tabla in ("clientes", "pedidos", "pagos"):
    r = resumen[tabla]
    print(f"{tabla:9} insertados={r['insertados']:>6} actualizados={r['actualizados']:>4} "
          f"cuarentena_nueva={r['cuarentena_nueva']:>4} bronze={'nuevo' if r['bronze_nuevo'] else 'ya existia'}")
print("dias de venta recalculados:", resumen["dias_recalculados"])


In [ ]:
# === 7. Verificación contra el ETL local (solo aplica con el conjunto de datos completo del caso) ===
ESPERADO = {"pedidos_validos": 19270, "pagos_validos": 20124, "clientes_validos": 6040,
            "filas_gold": 7925, "venta_neta": 14_177_333_600}
real = {
    "pedidos_validos": spark.table(T("silver_pedidos")).count(),
    "pagos_validos": spark.table(T("silver_pagos")).count(),
    "clientes_validos": spark.table(T("silver_clientes")).count(),
    "filas_gold": spark.table(T("gold_venta_neta_diaria")).count(),
    "venta_neta": spark.table(T("gold_venta_neta_diaria")).agg(F.sum("venta_neta")).first()[0],
}
for k, v in ESPERADO.items():
    print(f"{k:18} esperado={v:>16,} real={real[k]:>16,} {'OK' if real[k] == v else 'DIFERENTE'}")
assert real == ESPERADO, "Las cifras no coinciden con el ETL local"

display(spark.table(T("cuarentena")).groupBy("tabla_origen", "regla_violada")
        .agg(F.countDistinct("id_registro").alias("registros")).orderBy("tabla_origen", F.desc("registros")))
